# 03 — Manifest → priors → scale-conditioned mass regressor (GPU job 2)

The novel model: mass regression conditioned on measured metric geometry (docs/MODELS.md §4). Requires **notebook 01** (Nutrition5k on Drive).

Steps: (1) extract the geometry manifest from the RGB-D captures — the same quantities the phone measures; (2) fit the κ/φ/h̄ shape priors used by the pure-geometry pipeline; (3) train the regressor.

- **Runtime:** manifest extraction ~30–60 min (CPU-bound); training ~1–2 h on H100 at batch 128.
- **Benchmarks to beat:** 26.1% calorie MAPE (RGB) / 16.5% (RGB+depth).
- All artifacts and caches persist to Drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/physics-powered-portion-estimation'

# Framework caches stay LOCAL (fast, and mmap over Drive's FUSE mount is
# unreliable); the (small) pretrained backbone re-downloads in seconds.
# The DATASET and all OUTPUTS below live on Drive — that's what's worth keeping.
os.environ['HF_HOME'] = '/content/hf-cache'
os.environ['TORCH_HOME'] = '/content/torch-cache'

DATA = f'{DRIVE_ROOT}/data/nutrition5k'
OUT = f'{DRIVE_ROOT}/out'
CKPT = f'{DRIVE_ROOT}/checkpoints/mass-regressor.pt'
!mkdir -p "{OUT}"
!nvidia-smi | head -12

Mounted at /content/drive
Wed Jul  8 06:49:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   40C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+---------------------

In [2]:
# Clone the repo. Private repo → add your GitHub token as a Colab secret named
# GH_TOKEN (🔑 left sidebar). Public repo → no token needed. Fails LOUD.
import os, subprocess
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN') or ''
except Exception:
    pass
repo = 'github.com/Hakeem-Hannoon/physics-powered-portion-estimation.git'
url = f'https://x-access-token:{token}@{repo}' if token else f'https://{repo}'
subprocess.run(['rm', '-rf', '/content/ppe'])
r = subprocess.run(['git', 'clone', '--depth', '1', url, '/content/ppe'],
                   capture_output=True, text=True)
assert os.path.isdir('/content/ppe/model'), (
    (r.stderr.replace(token, '***') if token else r.stderr) +
    '\nClone failed — add a GH_TOKEN Colab secret (🔑) or make the repo public.')
print('repo ready')

%pip -q install timm pandas

repo ready


In [3]:
%cd /content/ppe
# 1. Geometry manifest (area, height, volume per dish + mass/kcal labels).
!python model/data/prepare_nutrition5k.py --root "{DATA}" --out "{OUT}/n5k-manifest.csv"
!head -3 "{OUT}/n5k-manifest.csv"

/content/ppe
Traceback (most recent call last):
  File "/content/ppe/model/data/prepare_nutrition5k.py", line 150, in <module>
    main()
  File "/content/ppe/model/data/prepare_nutrition5k.py", line 124, in main
    geometry = analyze_depth(depth)
               ^^^^^^^^^^^^^^^^^^^^
  File "/content/ppe/model/data/prepare_nutrition5k.py", line 43, in analyze_depth
    depth_raw = np.asarray(Image.open(depth_path), dtype=np.float32)
                           ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 3580, in open
    raise UnidentifiedImageError(msg)
PIL.UnidentifiedImageError: cannot identify image file '/content/drive/MyDrive/physics-powered-portion-estimation/data/nutrition5k/imagery/realsense_overhead/dish_1564159636/depth_raw.png'
head: cannot open '/content/drive/MyDrive/physics-powered-portion-estimation/out/n5k-manifest.csv' for reading: No such file or directory


In [4]:
# 2. Shape priors — the printed global κ replaces DEFAULT_KAPPA in
#    packages/pipeline/src/estimate.ts and feeds nutrition/'s shape_priors.
!python model/priors/fit_priors.py --manifest "{OUT}/n5k-manifest.csv" --out "{OUT}/priors.json"
!cat "{OUT}/priors.json"

Traceback (most recent call last):
  File "/content/ppe/model/priors/fit_priors.py", line 63, in <module>
    main()
  File "/content/ppe/model/priors/fit_priors.py", line 33, in main
    df = pd.read_csv(args.manifest)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1880, in _make_engine
    self.handles = get_handle(
 

In [5]:
# 3. Train. Per-epoch mass/kcal MAPE prints; best checkpoint saves to Drive.
!python model/train/mass_regressor_nutrition5k.py \
  --manifest "{OUT}/n5k-manifest.csv" \
  --epochs 50 --batch-size 128 \
  --output "{CKPT}"
print('README row template:')
print('| Nutrition5k test split | calorie MAPE | 26.1% RGB / 16.5% depth | **<best kcal MAPE from above>** |')

Traceback (most recent call last):
  File "/content/ppe/model/train/mass_regressor_nutrition5k.py", line 189, in <module>
    main()
  File "/content/ppe/model/train/mass_regressor_nutrition5k.py", line 134, in main
    manifest = pd.read_csv(args.manifest)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1880, in _